# Living Canvas: real-model test v3

**What v2 showed:**
- Image-to-image from the opening painting stays beautiful, but visitors barely change it. It keeps the opening's daylight colors, so night, snow and fog are out of reach unless the strength is high enough to warp the building.
- **The structure lock (SDXL-Turbo + Canny ControlNet, every frame from noise) was the only run whose changes followed the meaning:** winter turned pale and bare, bloom added purple blossoms. The changes were still too subtle.

**v3 drops image-to-image and makes the visitor signal stronger:**
1. **Expressiveness test.** Four extreme visitors on their own (night mist, snow, storm, golden crowd) × 5 settings:
   - **gain 1, 2, 3:** push further along the visitor's direction in embedding space
   - **guidance push 2, 3:** classifier-free guidance with the plain style prompt as the *negative*, so the model actively steers away from the opening toward the visitor
2. **"Your moment," then the crowd.** Each visitor gets two frames. In *your moment*, your direction takes over the painting. In *crowd after*, it settles into the crowd state, where you keep a share.
3. **Morph GIF** of the crowd run.

**Pass criteria:** (1) same chapel and pond, (2) **night looks like night, snow like snow**, (3) reads as a painting, (4) under about 3 s per frame.

**Setup:** Colab → *Runtime → Change runtime type → T4 GPU*, personal or Duke Google account. First load downloads about 9 GB. The expressiveness test takes about 2 minutes after that.

In [ ]:
!pip -q install "diffusers==0.31.0" "transformers==4.46.3" "accelerate==1.1.1"

In [ ]:
import torch, time, os, gc
import numpy as np, cv2
from PIL import Image, ImageDraw
assert torch.cuda.is_available(), "No GPU found. In Colab: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
os.makedirs("out", exist_ok=True)

In [ ]:
from diffusers import StableDiffusionXLControlNetPipeline, StableDiffusionXLPipeline, ControlNetModel, AutoencoderKL
t0 = time.time()
try:
    cn = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16, variant="fp16")
except Exception:
    cn = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
xl = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/sdxl-turbo", controlnet=cn, vae=vae, torch_dtype=torch.float16, variant="fp16").to("cuda")
plain = StableDiffusionXLPipeline.from_pipe(xl)   # same weights, no ControlNet: used once for the opening painting
xl.set_progress_bar_config(disable=True); plain.set_progress_bar_config(disable=True)
W, H = 768, 432
print(f"loaded in {time.time() - t0:.0f}s")

In [ ]:
STYLE = ("an impressionist oil painting by Claude Monet, a gothic chapel tower above a garden pond, "
         "reflections on the water, thick visible brushstrokes")
TONES = {"whisper": "hushed, soft, dim, quiet", "calm": "calm, still, gentle", "warm": "warm golden glow, tender",
         "excited": "bright, vivid, lively", "shouting": "loud, turbulent, dramatic, stormy"}
MOVES = {"float": "drifting, airy, floating mist", "glide": "smooth, serene, gliding", "dab": "light dabs of color, sparkling",
         "flick": "flickering, playful", "press": "heavy, pressing, dense", "wring": "twisting, wrung, tense",
         "punch": "forceful, explosive", "slash": "sweeping, slashing, windswept"}
VISITORS = [
    ("Finally submitted my thesis. I could sleep for a week.", "dusk settles, lilies open on still water", "calm", "float"),
    ("Everyone I love is in this room right now.", "golden light, the banks fill with people", "warm", "glide"),
    ("I am so stressed about the job market.", "a storm gathers over cold water", "shouting", "wring"),
    ("Missing my grandma tonight.", "night mist, an empty bank", "whisper", "press"),
    ("WE WON THE GAME!!", "a bright roar of color, crowds at the water", "shouting", "punch"),
    ("First snow of the year on the walk here.", "winter light on the tower, snow on the trees", "calm", "float"),
    ("Honestly no idea what I am doing with my life.", "thick fog swallows the far bank", "whisper", "wring"),
    ("Spring break in nine days!", "bright morning, the garden in full bloom", "excited", "flick"),
    ("I want to build something that matters.", "the air clears, the tower stands lit", "warm", "press"),
    ("Thinking about the people who are not here.", "lamplight in the windows, a quiet bank", "whisper", "glide"),
]
def visitor_prompt(reading, tone, move):
    return f"{STYLE}, {reading}, {TONES[tone]}, {MOVES[move]}"

@torch.no_grad()
def E(text):
    """SDXL conditioning for a prompt: token embeddings (1, 77, 2048) and pooled embedding (1, 1280)."""
    pe, _, pp, _ = xl.encode_prompt(prompt=text, device="cuda", num_images_per_prompt=1, do_classifier_free_guidance=False)
    return pe, pp

def gen():
    return torch.Generator("cuda").manual_seed(7)

S_PE, S_PP = E(STYLE)
DIRS = []   # each visitor's direction away from the plain style prompt
for _, r, t, m in VISITORS:
    pe, pp = E(visitor_prompt(r, t, m)); DIRS.append((pe - S_PE, pp - S_PP))

## Opening painting and its outlines (the structure lock)

In [ ]:
t0 = time.time()
opening = plain(prompt_embeds=S_PE, pooled_prompt_embeds=S_PP, num_inference_steps=4, guidance_scale=0.0,
                width=W, height=H, generator=gen()).images[0]
print(f"opening: {time.time() - t0:.2f}s"); display(opening)
edges = cv2.Canny(np.array(opening), 80, 160)
edge_img = Image.fromarray(np.stack([edges] * 3, -1)); display(edge_img.resize((W // 2, H // 2)))

In [ ]:
CN_SCALE = 0.45   # how tightly to hold the outlines: lower = freer, higher = stiffer

def decode(d_pe, d_pp, gain=1.0, push=0.0):
    """Frame for a direction (d_pe, d_pp) away from the style prompt.
    gain: multiply the direction in embedding space.
    push: classifier-free guidance with the plain style prompt as the negative (0 = off)."""
    kw = dict(prompt_embeds=S_PE + gain * d_pe, pooled_prompt_embeds=S_PP + gain * d_pp, image=edge_img,
              controlnet_conditioning_scale=CN_SCALE, num_inference_steps=4, width=W, height=H, generator=gen())
    if push > 1:
        kw.update(guidance_scale=push, negative_prompt_embeds=S_PE, negative_pooled_prompt_embeds=S_PP)
    else:
        kw.update(guidance_scale=0.0)
    return xl(**kw).images[0]

def grid(frames, labels, cols):
    w, h = frames[0].size; tw, th = w // 2, h // 2
    rows = (len(frames) + cols - 1) // cols
    G = Image.new("RGB", (cols * tw, rows * (th + 24)), "white"); d = ImageDraw.Draw(G)
    for i, (f, lab) in enumerate(zip(frames, labels)):
        x, y = (i % cols) * tw, (i // cols) * (th + 24)
        G.paste(f.resize((tw, th)), (x, y)); d.text((x + 4, y + th + 5), lab[:62], fill="black")
    return G

## 1 · Expressiveness test

Rows are visitors, columns are settings. **Pass:** at least one column where night looks like night, snow like snow, a storm like a storm and the golden crowd like a warm crowd, and it's still the same chapel. Pick that column's settings for step 2.

In [ ]:
SETTINGS = [("gain 1", 1, 0), ("gain 2", 2, 0), ("gain 3", 3, 0), ("push 2", 1, 2.0), ("push 3", 1, 3.0)]
TEST = [3, 5, 2, 1]   # night mist, snow, storm, golden crowd
frames, labels, times = [], [], []
for i in TEST:
    for name, g, p in SETTINGS:
        t = time.time(); frames.append(decode(*DIRS[i], gain=g, push=p)); times.append((name, time.time() - t))
        labels.append(f"{VISITORS[i][1][:30]} | {name}")
G = grid(frames, labels, cols=len(SETTINGS)); G.save("out/expressiveness_grid.jpg", quality=90); display(G)
for name in dict.fromkeys(n for n, _ in times):
    ts = [s for n, s in times if n == name]; print(f"{name}: {sum(ts) / len(ts):.2f}s per frame")

## 2 · Your moment, then the crowd

Set `GAIN` and `PUSH` from the winning column. Each visitor gets two frames:
- **you:** the crowd state pulled 85% of the way toward your direction (what you see while you're at the screen)
- **crowd:** the state after you settle in: `D ← D + α(d − D)`, where α is 0.25 to 0.5 depending on how different you are from the crowd

In [ ]:
GAIN, PUSH = 1, 3.0   # push 3 won the v3 expressiveness test (snow, storm, golden light)

def cos_dist(a, b):
    a, b = a.flatten().float(), b.flatten().float()
    return 1.0 if b.norm() < 1e-6 else 1 - torch.nn.functional.cosine_similarity(a, b, dim=0).item()

D_pe, D_pp = torch.zeros_like(S_PE), torch.zeros_like(S_PP)
frames, labels, crowd_states, times = [decode(D_pe, D_pp, GAIN, PUSH)], ["opening"], [(D_pe, D_pp)], []
for i, (d_pe, d_pp) in enumerate(DIRS):
    a = 0.25 + 0.25 * min(cos_dist(d_pe, D_pe), 1.0)
    t = time.time()
    you = decode(D_pe + 0.85 * (d_pe - D_pe), D_pp + 0.85 * (d_pp - D_pp), GAIN, PUSH)
    D_pe, D_pp = D_pe + a * (d_pe - D_pe), D_pp + a * (d_pp - D_pp)
    crowd = decode(D_pe, D_pp, GAIN, PUSH); times.append((time.time() - t) / 2)
    crowd_states.append((D_pe.clone(), D_pp.clone()))
    frames += [you, crowd]; labels += [f"{i + 1} YOU: {VISITORS[i][1][:34]}", f"{i + 1} crowd after (a={a:.2f})"]
G = grid(frames, labels, cols=4); G.save("out/moment_and_crowd_grid.jpg", quality=90); display(G)
frames[0].save("out/moment_and_crowd.gif", save_all=True, append_images=frames[1:], duration=1100, loop=0)
print(f"median {sorted(times)[len(times) // 2]:.2f}s per frame")

## 3 · Morph: the crowd state blending smoothly between visitors (the booth's idle screen)

In [ ]:
TWEENS, morph, t0 = 4, [], time.time()
for (a_pe, a_pp), (b_pe, b_pp) in zip(crowd_states, crowd_states[1:]):
    for k in range(1, TWEENS + 1):
        f = k / TWEENS; morph.append(decode(a_pe + f * (b_pe - a_pe), a_pp + f * (b_pp - a_pp), GAIN, PUSH))
morph[0].save("out/crowd_morph.gif", save_all=True, append_images=morph[1:], duration=150, loop=0)
print(f"{len(morph)} frames, {(time.time() - t0) / len(morph):.2f}s each -> out/crowd_morph.gif")

## 4 · Night and painterliness

v3 findings: push 3 reached snow, storm and golden light, but not night. SDXL-Turbo also drifts toward a flat gouache-poster look.
- **Night:** a stronger night description, and pushes up to 5.
- **Painterliness:** the guidance's negative becomes the style *plus* "flat illustration, poster art", so the push moves away from the poster look as well as toward the visitor.

Columns: the v3 push 3 for reference, then the new negative at push 3, 4 and 5.

In [ ]:
NEG_PE, NEG_PP = E(STYLE + ", flat illustration, poster art, cartoon, smooth vector shading")

def decode2(d_pe, d_pp, push=3.0):
    """Like decode with push, but the negative is a flat-poster version of the style,
    so guidance pushes toward the visitor AND away from the poster look."""
    return xl(prompt_embeds=S_PE + d_pe, pooled_prompt_embeds=S_PP + d_pp,
              negative_prompt_embeds=NEG_PE, negative_pooled_prompt_embeds=NEG_PP,
              image=edge_img, controlnet_conditioning_scale=CN_SCALE, num_inference_steps=4,
              guidance_scale=push, width=W, height=H, generator=gen()).images[0]

TRY = [("night mist, an empty bank", "whisper", "press"),
       ("deep blue night, moonlight on the water, dark sky, lamplit windows", "whisper", "press"),
       ("winter light on the tower, snow on the trees", "calm", "float")]
frames, labels, times = [], [], []
for reading, t, m in TRY:
    pe, pp = E(visitor_prompt(reading, t, m)); d = (pe - S_PE, pp - S_PP)
    frames.append(decode(*d, push=3.0)); labels.append(f"{reading[:30]} | v3 push 3")
    for p in (3.0, 4.0, 5.0):
        t0 = time.time(); frames.append(decode2(*d, push=p)); times.append(time.time() - t0)
        labels.append(f"{reading[:30]} | new neg, push {p:g}")
G = grid(frames, labels, cols=4); G.save("out/night_and_painterly.jpg", quality=90); display(G)
print(f"push frames: {sum(times) / len(times):.2f}s each (guidance runs the model twice per step)")

## Download

In [ ]:
!zip -qr living_canvas_v3_results.zip out
try:
    from google.colab import files
    files.download("living_canvas_v3_results.zip")
except ImportError:
    print("Not in Colab: results are in ./out and living_canvas_v3_results.zip")